# پروژه رگرسیون: پیش‌بینی قیمت آپارتمان در تهران
مراحل: بارگذاری داده، پاک‌سازی، تحلیل اکتشافی، مدل‌سازی (Linear / Ridge / Random Forest / Gradient Boosting) و ارزیابی.

In [ ]:
import pandas as pd, numpy as np, matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression, Ridge
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
df = pd.read_csv('housePrice.csv')
print(df.shape)
df.head()

## ۱. بررسی اولیه داده

In [ ]:
df.info()
print(df.isna().sum())

## ۲. پاک‌سازی داده
- ستون `Area` به‌صورت رشته (با کاما و فاصله) ذخیره شده و باید عددی شود.
- حذف سطرهای بدون آدرس.
- حذف متراژهای اشتباه (مقدارهای بسیار بزرگ مثل ۱۶,۱۶۰,۰۰۰,۰۰۰ یا ۳۶۰۰ متر).
- حذف سطرهای تکراری.

In [ ]:
df['Area'] = pd.to_numeric(df['Area'].astype(str).str.replace(',', '').str.strip(), errors='coerce')
n0 = len(df)
df = df.dropna(subset=['Address', 'Area'])
print('بعد از حذف آدرس خالی:', len(df))
df = df[df['Area'] <= 1000]   # متراژ غیرواقعی
print('بعد از حذف متراژ غیرواقعی:', len(df))
df = df.drop_duplicates()
print('بعد از حذف تکراری‌ها:', len(df), '| کل حذف‌شده:', n0 - len(df))
df[['Area', 'Price(USD)']].describe()

## ۳. تحلیل اکتشافی

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
ax[0].hist(df['Area'], bins=50); ax[0].set_title('Area')
ax[1].hist(df['Price(USD)'], bins=50); ax[1].set_title('Price(USD)')
ax[2].scatter(df['Area'], df['Price(USD)'], alpha=.3); ax[2].set_title('Area vs Price')
plt.tight_layout(); plt.show()
print(df.groupby('Address')['Price(USD)'].median().sort_values(ascending=False).head(10))
print(df[['Area','Room','Parking','Warehouse','Elevator','Price(USD)']].astype(float).corr()['Price(USD)'])

## ۴. مدل‌سازی
هدف قیمت به دلار است. چون توزیع قیمت چوله است روی `log(Price)` آموزش می‌دهیم و نتیجه را به مقیاس اصلی برمی‌گردانیم. ستون `Price` (تومان) هم‌ارز هدف است و برای جلوگیری از نشت داده از ویژگی‌ها کنار گذاشته می‌شود.

In [ ]:
X = df[['Area','Room','Parking','Warehouse','Elevator','Address']].copy()
for col in ['Parking','Warehouse','Elevator']:
    X[col] = X[col].astype(int)
y = np.log(df['Price(USD)'])
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
pre = ColumnTransformer([('addr', OneHotEncoder(handle_unknown='ignore'), ['Address'])], remainder='passthrough')
models = {
 'Linear Regression': LinearRegression(),
 'Ridge': Ridge(alpha=1.0),
 'Random Forest': RandomForestRegressor(n_estimators=300, random_state=42, n_jobs=-1),
 'Gradient Boosting': GradientBoostingRegressor(n_estimators=300, learning_rate=0.05, max_depth=4, random_state=42),
}
rows, fitted = [], {}
for name, m in models.items():
    p = Pipeline([('pre', pre), ('m', m)]).fit(X_train, y_train)
    pred = np.exp(p.predict(X_test)); true = np.exp(y_test)
    fitted[name] = p
    rows.append({'Model': name,
                 'R2 (log)': r2_score(y_test, p.predict(X_test)),
                 'R2 (USD)': r2_score(true, pred),
                 'MAE (USD)': mean_absolute_error(true, pred),
                 'RMSE (USD)': np.sqrt(mean_squared_error(true, pred)),
                 'CV R2 (log)': cross_val_score(Pipeline([('pre', pre), ('m', m)]), X, y, cv=5, scoring='r2').mean()})
res = pd.DataFrame(rows).sort_values('R2 (log)', ascending=False).round(3)
res

## ۵. بهترین مدل: مقدار واقعی در برابر پیش‌بینی

In [ ]:
best = res.iloc[0]['Model']; print('Best model:', best)
pred = np.exp(fitted[best].predict(X_test)); true = np.exp(y_test)
plt.figure(figsize=(5, 5)); plt.scatter(true, pred, alpha=.4)
lim = [true.min(), true.max()]; plt.plot(lim, lim, 'r--')
plt.xscale('log'); plt.yscale('log'); plt.xlabel('Actual USD'); plt.ylabel('Predicted USD'); plt.title(best); plt.show()

## ۶. نتیجه‌گیری
متراژ و محله مهم‌ترین عوامل قیمت‌اند. جدول بخش ۴ مدل‌ها را مقایسه می‌کند و بهترین مدل بر اساس R² انتخاب شده است.